# Actividad 9 – Análisis de entregas y SLA
### Iván López - A01284875

## Metodología
* Leer el archivo csv con PySpark.
* Calcular horas de entrega
* Marcar cumplimiento de SLA
* Calcular KPIs por ruta
* Exploración y análisis 

## Librerías requeridas

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

## Configuración de Spark

In [2]:
# Crear sesión Spark
try:
    sc.stop()
except Exception:
    pass

In [3]:
spark = (SparkSession.builder
         .appName("NotebookSession")
         .master("local[*]")
         .config("spark.ui.port", "0")
         .getOrCreate())

sc = spark.sparkContext
sc.setLogLevel("WARN")

print(spark.version, sc.appName)

4.0.1 NotebookSession


## Carga de datos

In [5]:
# leer el CSV
df_del = spark.read.csv('activity9_deliveries_big.csv', header=True, inferSchema=True) 

In [6]:
df_del = (df_del
          .withColumn('created_ts', F.to_timestamp('created_ts'))
          .withColumn('shipped_ts', F.to_timestamp('shipped_ts'))
          .withColumn('delivered_ts', F.to_timestamp('delivered_ts')))

In [7]:
# verificar esquema
df_del.printSchema()

root
 |-- shipment_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- created_ts: timestamp (nullable = true)
 |-- shipped_ts: timestamp (nullable = true)
 |-- delivered_ts: timestamp (nullable = true)
 |-- origin_city: string (nullable = true)
 |-- dest_city: string (nullable = true)
 |-- status: string (nullable = true)



In [8]:
#primeras filas
df_del.show(10)

+--------------------+-----------+-------------------+-------------------+-------------------+-----------+-----------+-------+
|         shipment_id|customer_id|         created_ts|         shipped_ts|       delivered_ts|origin_city|  dest_city| status|
+--------------------+-----------+-------------------+-------------------+-------------------+-----------+-----------+-------+
|1a3de4de-215d-485...|      44922|2025-05-11 00:00:00|2025-05-11 19:00:00|2025-05-12 22:00:00|       CDMX|       CDMX|ON_TIME|
|c4a1a801-d2d1-4ed...|      91077|2025-05-10 13:00:00|2025-05-10 20:00:00|2025-05-11 12:00:00|       León|       CDMX|ON_TIME|
|50e719a8-94f0-4ef...|      79945|2025-05-04 08:00:00|2025-05-04 20:00:00|2025-05-05 19:00:00|     Puebla|     Puebla|ON_TIME|
|0ea7e329-dd40-484...|      77747|2025-05-26 10:00:00|2025-05-27 10:00:00|2025-05-27 20:00:00|Guadalajara|Guadalajara|ON_TIME|
|9cbf3977-84ae-485...|      63836|2025-05-22 20:00:00|2025-05-24 04:00:00|2025-05-27 16:00:00|    Tijuana|  Mon

## Agregar columna horas de entrega

In [10]:
#columna horas de entrega
df_del=(df_del
        .withColumn('hours_to_delivery', 
                    (F.unix_timestamp('delivered_ts')-F.unix_timestamp('created_ts'))/3600))

df_del.show(10)

+--------------------+-----------+-------------------+-------------------+-------------------+-----------+-----------+-------+-----------------+
|         shipment_id|customer_id|         created_ts|         shipped_ts|       delivered_ts|origin_city|  dest_city| status|hours_to_delivery|
+--------------------+-----------+-------------------+-------------------+-------------------+-----------+-----------+-------+-----------------+
|1a3de4de-215d-485...|      44922|2025-05-11 00:00:00|2025-05-11 19:00:00|2025-05-12 22:00:00|       CDMX|       CDMX|ON_TIME|             46.0|
|c4a1a801-d2d1-4ed...|      91077|2025-05-10 13:00:00|2025-05-10 20:00:00|2025-05-11 12:00:00|       León|       CDMX|ON_TIME|             23.0|
|50e719a8-94f0-4ef...|      79945|2025-05-04 08:00:00|2025-05-04 20:00:00|2025-05-05 19:00:00|     Puebla|     Puebla|ON_TIME|             35.0|
|0ea7e329-dd40-484...|      77747|2025-05-26 10:00:00|2025-05-27 10:00:00|2025-05-27 20:00:00|Guadalajara|Guadalajara|ON_TIME|    

## Agregar columna de cumplimiento de SLA

In [11]:
df_del=(df_del
        .withColumn('sla_ok', 
                    (F.when(F.col('hours_to_delivery') <= 72, 1).otherwise(0))))

df_del.show(10)

+--------------------+-----------+-------------------+-------------------+-------------------+-----------+-----------+-------+-----------------+------+
|         shipment_id|customer_id|         created_ts|         shipped_ts|       delivered_ts|origin_city|  dest_city| status|hours_to_delivery|sla_ok|
+--------------------+-----------+-------------------+-------------------+-------------------+-----------+-----------+-------+-----------------+------+
|1a3de4de-215d-485...|      44922|2025-05-11 00:00:00|2025-05-11 19:00:00|2025-05-12 22:00:00|       CDMX|       CDMX|ON_TIME|             46.0|     1|
|c4a1a801-d2d1-4ed...|      91077|2025-05-10 13:00:00|2025-05-10 20:00:00|2025-05-11 12:00:00|       León|       CDMX|ON_TIME|             23.0|     1|
|50e719a8-94f0-4ef...|      79945|2025-05-04 08:00:00|2025-05-04 20:00:00|2025-05-05 19:00:00|     Puebla|     Puebla|ON_TIME|             35.0|     1|
|0ea7e329-dd40-484...|      77747|2025-05-26 10:00:00|2025-05-27 10:00:00|2025-05-27 20:

## KPIs por ruta

In [27]:
df_kpis=(df_del.groupBy('origin_city', 'dest_city')
         .agg(
             F.count('*').alias('n_shipments'),
             F.round(F.avg('hours_to_delivery'),2).alias('avg_hours'),
             F.round(F.avg('sla_ok'), 2).alias('sla_rate'))
            .orderBy(F.desc('sla_rate')))

df_kpis.show()

+-----------+-----------+-----------+---------+--------+
|origin_city|  dest_city|n_shipments|avg_hours|sla_rate|
+-----------+-----------+-----------+---------+--------+
|Guadalajara|Guadalajara|       1107|    85.59|     0.4|
|Guadalajara|     Puebla|       1115|    87.09|    0.39|
|    Tijuana|Guadalajara|       1146|    86.79|    0.38|
|       León|Guadalajara|       1139|    88.13|    0.38|
|       CDMX|Guadalajara|       1164|    86.82|    0.38|
|  Monterrey|       León|       1075|    87.75|    0.38|
|       CDMX|     Puebla|       1109|    88.05|    0.38|
|       León|  Monterrey|       1100|    87.58|    0.38|
|Guadalajara|       León|       1126|    86.77|    0.38|
|     Puebla|    Tijuana|       1138|    88.53|    0.37|
|     Puebla|Guadalajara|       1155|    87.36|    0.37|
|       CDMX|       CDMX|       1101|    87.86|    0.37|
|     Puebla|       León|       1081|    87.69|    0.37|
|     Puebla|       CDMX|       1123|    87.79|    0.37|
|    Tijuana|       León|      

## Exploración de rutas con peor cumplimento de SLA

In [28]:
df_kpis.orderBy(F.asc('sla_rate')).show()

+-----------+-----------+-----------+---------+--------+
|origin_city|  dest_city|n_shipments|avg_hours|sla_rate|
+-----------+-----------+-----------+---------+--------+
|       León|       CDMX|       1102|    89.62|    0.33|
|       CDMX|    Tijuana|       1099|    90.64|    0.34|
|Guadalajara|    Tijuana|       1110|    90.21|    0.34|
|    Tijuana|       CDMX|       1102|    89.23|    0.34|
|       León|       León|       1094|    89.02|    0.34|
|     Puebla|  Monterrey|       1087|    88.57|    0.35|
|    Tijuana|     Puebla|       1112|    89.28|    0.35|
|  Monterrey|Guadalajara|       1099|    88.31|    0.35|
|Guadalajara|       CDMX|       1140|    89.18|    0.35|
|  Monterrey|  Monterrey|       1035|    89.02|    0.35|
|     Puebla|     Puebla|       1099|    89.41|    0.35|
|       CDMX|  Monterrey|       1167|    88.73|    0.35|
|       León|     Puebla|       1096|    89.39|    0.35|
|       CDMX|       León|       1130|    90.29|    0.35|
|    Tijuana|       León|      

## Análisis
Las 5 rutas con peor cumplimiento de SLA son:
* León -> CDMX
* CDMX -> Tijuana
* Guadalajara -> Tijuana
* Tijuana -> CDMX
* León -> León

bajo la premisa de que ni siquiera alcanzan el 35% de cumplimiento de SLA que es donde otras rutas ya se empiezan a equiparar más, por lo cual estas son las que necesitan mayor atención. Esto principalmente en el caso de los envíos que salen de León dado que los que llegan y salen de Tijuana se entiende que sea por su ubicación geográfica un tanto lejana, pero para León no se comprende si el problema radique en que haya alguna ineficiencia operativa al cargar el vehículo para el inicio del envío desde el punto de origen o si se trate de alguna complicación en el trayecto.

Sin embargo, observando los KPIs para el resto de rutas se puede concluir que realmente no existen diferencias significativas entre ciudades, pues todas las rutas presentan un desempeño crítico con tasas SLA bastante bajas, que van apenas entre el 33% y el 40%. Para mejorar esto, más allá de optimizar la logística ya sea en frontera, centro o norte, es urgente recalibrar la promesa de tiempo de entrega ya que un cumplimiento tan bajo sugiere que el tiempo objetivo del SLA (72 horas) es poco realista para la capacidad operativa actual que va de 85 a 90 horas aproximadamente en cuanto a promedio.